# 04. Tools & Structured Outputs

The model does NOT call your backend directly. Tool calling is a mechanism where the model **proposes** a structured action, and your application **validates, authorizes, and executes** it.

In this lab, we will build a secure execution boundary for a customer support bot that can refund orders.

## Part 1 — Raw JSON Schema
To give a model a tool, we provide a JSON Schema that describes the function name, purpose, and required arguments.

In [1]:
raw_schema = {
    'type': 'function',
    'function': {
        'name': 'get_order',
        'description': 'Retrieves details about a specific customer order.',
        'parameters': {
            'type': 'object',
            'properties': {
                'order_id': {
                    'type': 'string',
                    'description': 'The unique identifier for the order (e.g. ORD-123)'
                }
            },
            'required': ['order_id'],
            'additionalProperties': False
        }
    }
}
print('Raw schema defined.')

Raw schema defined.


## Part 2 — Pydantic
Manually writing JSON Schemas is error-prone. Modern Python uses Pydantic to define typed models, which can automatically generate schemas.

**Best Practice:** Never use `float` for currency. We use `int` for cents.

In [2]:
from pydantic import BaseModel, Field, ConfigDict
from enum import Enum

class RefundReason(str, Enum):
    DAMAGED = 'damaged'
    LOST = 'lost'
    CUSTOMER_REQUEST = 'customer_request'

class IssueRefundArgs(BaseModel):
    """Issues a refund for a customer order. Requires authorization."""
    model_config = ConfigDict(extra='forbid')

    order_id: str = Field(..., description='The unique order identifier.')
    amount_cents: int = Field(..., gt=0, description='Amount to refund in cents (must be positive).')
    reason: RefundReason = Field(..., description='The approved reason for the refund.')

print('Generated JSON Schema for IssueRefundArgs:')
import json
print(json.dumps(IssueRefundArgs.model_json_schema(), indent=2))

Generated JSON Schema for IssueRefundArgs:
{
  "$defs": {
    "RefundReason": {
      "enum": [
        "damaged",
        "lost",
        "customer_request"
      ],
      "title": "RefundReason",
      "type": "string"
    }
  },
  "additionalProperties": false,
  "description": "Issues a refund for a customer order. Requires authorization.",
  "properties": {
    "order_id": {
      "description": "The unique order identifier.",
      "title": "Order Id",
      "type": "string"
    },
    "amount_cents": {
      "description": "Amount to refund in cents (must be positive).",
      "exclusiveMinimum": 0,
      "title": "Amount Cents",
      "type": "integer"
    },
    "reason": {
      "$ref": "#/$defs/RefundReason",
      "description": "The approved reason for the refund."
    }
  },
  "required": [
    "order_id",
    "amount_cents",
    "reason"
  ],
  "title": "IssueRefundArgs",
  "type": "object"
}


## Part 3 — Valid and Invalid Tool Arguments
Let's simulate parsing arguments generated by a model. If the model hallucinates or provides invalid data types, Pydantic catches it.

In [3]:
from pydantic import ValidationError

valid_json = '{"order_id": "ORD-123", "amount_cents": 5000, "reason": "damaged"}'
invalid_json_negative_amount = '{"order_id": "ORD-123", "amount_cents": -100, "reason": "damaged"}'
invalid_json_extra_field = '{"order_id": "ORD-123", "amount_cents": 5000, "reason": "damaged", "note": "refund please"}'

print('--- Valid Request ---')
print(IssueRefundArgs.model_validate_json(valid_json))

print('\n--- Invalid Negative Amount ---')
try:
    IssueRefundArgs.model_validate_json(invalid_json_negative_amount)
except ValidationError as e:
    print('Caught ValidationError:', e.errors()[0]['msg'])

print('\n--- Invalid Extra Field ---')
try:
    IssueRefundArgs.model_validate_json(invalid_json_extra_field)
except ValidationError as e:
    print('Caught ValidationError:', e.errors()[0]['msg'])

--- Valid Request ---
order_id='ORD-123' amount_cents=5000 reason=<RefundReason.DAMAGED: 'damaged'>

--- Invalid Negative Amount ---
Caught ValidationError: Input should be greater than 0

--- Invalid Extra Field ---
Caught ValidationError: Extra inputs are not permitted


## Part 4 — Mocked Model Tool Call
Different providers (OpenAI, Anthropic) use different JSON structures to represent a tool call. We normalize them into a provider-neutral internal object.

In [4]:
class ToolCall(BaseModel):
    id: str
    name: str
    arguments: str

mocked_tool_call = ToolCall(
    id='call_abc123',
    name='issue_refund',
    arguments='{"order_id": "ORD-456", "amount_cents": 1000, "reason": "customer_request"}'
)
print('Normalized Tool Call:', mocked_tool_call)

Normalized Tool Call: id='call_abc123' name='issue_refund' arguments='{"order_id": "ORD-456", "amount_cents": 1000, "reason": "customer_request"}'


## Part 5 — Tool Registry & Safe Dispatch
We define our actual Python functions and register them. If the model asks for a tool not in the registry, we fail closed.

In [5]:
def get_order_impl(order_id: str):
    return {'order_id': order_id, 'status': 'delivered', 'total_cents': 5000}

def issue_refund_impl(order_id: str, amount_cents: int, reason: RefundReason):
    return {'status': 'refunded', 'amount': amount_cents, 'order_id': order_id}

TOOL_REGISTRY = {
    'get_order': get_order_impl,
    'issue_refund': issue_refund_impl
}

def dispatch_tool(tool_name: str, args_dict: dict):
    if tool_name not in TOOL_REGISTRY:
        raise ValueError(f'Security Exception: Unknown tool {tool_name}')
    return TOOL_REGISTRY[tool_name](**args_dict)

print('Executing known tool:', dispatch_tool('get_order', {'order_id': 'ORD-123'}))
try:
    dispatch_tool('delete_database', {})
except ValueError as e:
    print('Caught Unknown Tool:', e)

Executing known tool: {'order_id': 'ORD-123', 'status': 'delivered', 'total_cents': 5000}
Caught Unknown Tool: Security Exception: Unknown tool delete_database


## Part 6 — Execution Context
The model is untrusted and cannot dictate who is calling the tool. Identity and permissions come from the application's Execution Context.

In [6]:
class ExecutionContext(BaseModel):
    user_id: str
    tenant_id: str
    roles: set[str]

context = ExecutionContext(user_id='agent_007', tenant_id='tenant_a', roles={'support_agent'})
print('Active Context:', context)

Active Context: user_id='agent_007' tenant_id='tenant_a' roles={'support_agent'}


## Parts 7, 8 & 9 — Schema, Business, and Authorization Validation
Let's combine Schema Validation (Pydantic), Business Validation (refund <= total), and Authorization (role check).

In [7]:
def safe_issue_refund(ctx: ExecutionContext, raw_args: str):
    # 7. Schema Validation
    try:
        args = IssueRefundArgs.model_validate_json(raw_args)
    except ValidationError as e:
        return {'error': 'SCHEMA_ERROR', 'details': str(e)}

    # 8. Business Validation
    order = get_order_impl(args.order_id)
    if args.amount_cents > order['total_cents']:
        return {'error': 'BUSINESS_ERROR', 'details': 'Refund exceeds order total.'}

    # 9. Authorization
    if 'support_agent' not in ctx.roles and 'admin' not in ctx.roles:
        return {'error': 'AUTH_ERROR', 'details': 'Permission denied.'}

    # Execution
    return issue_refund_impl(args.order_id, args.amount_cents, args.reason)

print('Valid refund:', safe_issue_refund(context, '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged"}'))
print('Business violation:', safe_issue_refund(context, '{"order_id": "ORD-123", "amount_cents": 99999, "reason": "damaged"}'))
bad_context = ExecutionContext(user_id='hacker', tenant_id='tenant_a', roles={'guest'})
print('Auth violation:', safe_issue_refund(bad_context, '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged"}'))

Valid refund: {'status': 'refunded', 'amount': 1000, 'order_id': 'ORD-123'}
Business violation: {'error': 'BUSINESS_ERROR', 'details': 'Refund exceeds order total.'}
Auth violation: {'error': 'AUTH_ERROR', 'details': 'Permission denied.'}


## Parts 10 & 11 — Read vs Write & Idempotency
Consequential writes (`issue_refund`) require an idempotency key to prevent double processing if the model retries the tool.

In [8]:
PROCESSED_REFUNDS = set()

def idempotent_refund(idempotency_key: str, order_id: str):
    if idempotency_key in PROCESSED_REFUNDS:
        return {'status': 'already_processed', 'order_id': order_id}
    PROCESSED_REFUNDS.add(idempotency_key)
    return {'status': 'refunded', 'order_id': order_id}

print('First call:', idempotent_refund('req_111', 'ORD-999'))
print('Duplicate call:', idempotent_refund('req_111', 'ORD-999'))

First call: {'status': 'refunded', 'order_id': 'ORD-999'}
Duplicate call: {'status': 'already_processed', 'order_id': 'ORD-999'}


## Parts 12 & 13 — Error Taxonomy & Bounded Correction
We allow the model to correct Schema errors, but we do NOT retry Authorization errors. Retries must be bounded.

In [9]:
def simulate_correction_loop(raw_args: str):
    max_retries = 2
    for attempt in range(max_retries):
        print(f'Attempt {attempt+1}...')
        result = safe_issue_refund(context, raw_args)
        if 'error' in result:
            if result['error'] == 'SCHEMA_ERROR':
                print('Schema error. Instructing model to fix parameters.')
                # Simulate model fixing the JSON on next turn...
                raw_args = '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged"}'
            elif result['error'] == 'AUTH_ERROR':
                print('Auth error. Halting loop immediately. No retries.')
                break
            else:
                print(f"Business error: {result['error']}. Stopping.")
                break
        else:
            print('Success!', result)
            break

print('--- Recoverable Schema Error ---')
simulate_correction_loop('{"order_id": "ORD-123"}') # Missing fields
print('\n--- Unrecoverable Auth Error ---')
safe_issue_refund = lambda ctx, args: {'error': 'AUTH_ERROR'} # Mock auth failure
simulate_correction_loop('{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged"}')

--- Recoverable Schema Error ---
Attempt 1...
Schema error. Instructing model to fix parameters.
Attempt 2...
Success! {'status': 'refunded', 'amount': 1000, 'order_id': 'ORD-123'}

--- Unrecoverable Auth Error ---
Attempt 1...
Auth error. Halting loop immediately. No retries.


## Parts 14 & 15 — Structured Tool Results & Final Output
Instead of returning raw text from tools, return structured data. 
Additionally, the model's *final* answer can also be structured (`SupportDecision`).

In [10]:
from typing import Literal

class SupportDecision(BaseModel):
    category: Literal['refund', 'replacement', 'escalate']
    summary: str
    requires_human: bool

model_final_output = '{"category": "refund", "summary": "Approved $10 refund for damaged item.", "requires_human": false}'
decision = SupportDecision.model_validate_json(model_final_output)
print('Parsed final structured output:', decision)

Parsed final structured output: category='refund' summary='Approved $10 refund for damaged item.' requires_human=False


## Parts 16 & 17 — Multiple Tools & Selection Evaluation
If you provide `get_order`, `get_policy`, and `issue_refund`, the model must choose the right one. We evaluate accuracy.

In [11]:
evaluation_cases = [
    {'prompt': 'Where is my order?', 'expected_tool': 'get_order', 'model_choice': 'get_order'},
    {'prompt': 'I want a refund.', 'expected_tool': 'get_policy', 'model_choice': 'issue_refund'}, # Model jumped the gun!
]
correct = sum(1 for case in evaluation_cases if case['expected_tool'] == case['model_choice'])
print(f"Tool Selection Accuracy: {correct}/{len(evaluation_cases)} ({(correct/len(evaluation_cases))*100}%)")

Tool Selection Accuracy: 1/2 (50.0%)


## Part 18 — Optional Real OpenAI Tool-Calling Example
This section uses the official `openai` SDK to execute a real tool call end-to-end, passing the schema, receiving the proposal, executing the mock function, and returning the result.

In [12]:
import os
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    from openai import OpenAI
    import json
    client = OpenAI()

    # 1. Define tool schema for OpenAI
    tools = [{
        'type': 'function',
        'function': {
            'name': 'get_order',
            'description': 'Retrieves details about a specific customer order.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'order_id': {'type': 'string'}
                },
                'required': ['order_id'],
                'additionalProperties': False
            }
        }
    }]

    messages = [{'role': 'user', 'content': 'Where is order ORD-1001?'}]

    # 2. Call model (forces tool call if needed)
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
        tools=tools
    )
    msg = response.choices[0].message
    messages.append(msg)

    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        print(f"Model proposed calling: {tool_call.function.name} with {tool_call.function.arguments}")

        # 3. Application executes (using our mock)
        args = json.loads(tool_call.function.arguments)
        result = {'order_id': args.get('order_id'), 'status': 'shipped', 'delivery_date': 'tomorrow'}

        # 4. Send result back
        messages.append({
            'role': 'tool',
            'tool_call_id': tool_call.id,
            'content': json.dumps(result)
        })

        # 5. Get final answer
        final_response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages
        )
        print("\nFinal Model Response:", final_response.choices[0].message.content)
else:
    print('No OPENAI_API_KEY found. Skipping real API call.')

No OPENAI_API_KEY found. Skipping real API call.


## Parts 19 & 20 — Optional Real OpenAI Structured Output
We can also use the `.parse()` API (Structured Outputs) to guarantee the model replies with our Pydantic model (`SupportDecision`).

In [13]:
if api_key:
    try:
        # Using the beta.messages.parse or beta.chat.completions.parse API
        response = client.beta.chat.completions.parse(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': 'I want to escalate this broken TV!'}],
            response_format=SupportDecision,
        )
        decision_obj = response.choices[0].message.parsed
        print('Real Structured Output object:', decision_obj)
        print('Category extracted:', decision_obj.category)
    except AttributeError:
        print('Update openai SDK to use .parse() for structured outputs.')
else:
    print('No OPENAI_API_KEY found. Skipping real API call.')

No OPENAI_API_KEY found. Skipping real API call.
